# Load files

In [1]:
import json

with open("../CSVs/relations_final.json", "r") as f:
    relations = json.load(f)
print(len(relations))

360026


In [2]:
count = 0
for k, v in relations.items():
    count += 1
    if 10 < count and count < 20: 
        print(k, v)
    

@CHEMICAL_Phytol|Association|@CHEMICAL_Vitamin_K_1 {'r1': 'D010836', 'r2': 'D010837', 'type': 'Association', 'r1accession': '@CHEMICAL_Phytol', 'r2accession': '@CHEMICAL_Vitamin_K_1', 'pmids': ['38124486'], 'score': ['0.9721'], 'id': ['@CHEMICAL_Phytol', 'Association', '@CHEMICAL_Vitamin_K_1']}
@CHEMICAL_Homogentisic_Acid|Positive_Correlation|@CHEMICAL_Tocotrienols {'r1': 'D006713', 'r2': 'D024508', 'type': 'Positive_Correlation', 'r1accession': '@CHEMICAL_Homogentisic_Acid', 'r2accession': '@CHEMICAL_Tocotrienols', 'pmids': ['38124486'], 'score': ['0.7383'], 'id': ['@CHEMICAL_Homogentisic_Acid', 'Positive_Correlation', '@CHEMICAL_Tocotrienols']}
@CHEMICAL_Chlorophyll|Positive_Correlation|@CHEMICAL_Phytol {'r1': 'D002734', 'r2': 'D010836', 'type': 'Positive_Correlation', 'r1accession': '@CHEMICAL_Chlorophyll', 'r2accession': '@CHEMICAL_Phytol', 'pmids': ['40082754'], 'score': ['0.8202']}
@CHEMICAL_Tocopherols|Association|@GENE_VTE5 {'r1': 'D024505', 'r2': 830328, 'type': 'Association',

In [3]:
with open("../CSVs/entities_final.json", "r") as fe:
    entities = json.load(fe)
print(len(entities))

68702


In [4]:
count = 0
for k, v in entities.items():
    count += 1
    if count < 10: 
        print(k, v)

D001471 {'id': 'D001471', 'name': 'Barrett Esophagus', 'type': 'disease', 'pmids': ['40716632'], 'accession': '@DISEASE_Barrett_Esophagus', 'subtype': 'unknown', 'gene_ids': 'unknown'}
D000782 {'id': 'D000782', 'name': 'Aneuploidy', 'type': 'disease', 'pmids': ['40488668'], 'accession': '@DISEASE_Aneuploidy', 'subtype': 'unknown', 'gene_ids': 'unknown'}
C005072 {'id': 'C005072', 'name': '5-doxylstearic acid', 'type': 'chemical', 'pmids': ['40895728'], 'accession': '@CHEMICAL_5_doxylstearic_acid', 'subtype': 'unknown', 'gene_ids': 'unknown'}
D008228 {'id': 'D008228', 'name': 'Lymphoma Non-Hodgkin', 'type': 'disease', 'pmids': ['40484291', '40927528', '40654923', '40461678', '36572783', '40393126', '16930780', '39800921'], 'accession': '@DISEASE_Lymphoma_Non_Hodgkin', 'subtype': 'unknown', 'gene_ids': 'unknown'}
D015792 {'id': 'D015792', 'name': 'Retinal Dysplasia', 'type': 'disease', 'pmids': ['38621974', '40792210', '38612746'], 'accession': '@DISEASE_Retinal_Dysplasia', 'subtype': 'un

# Build extended graph

In [5]:
def prepareForVisualization(entitydict, relationdict):
    refinedrelations = dict()
    unique_relationtypes = set()
    unique_entitytypes = set()
    genesWithVariants = set()

    for trip, rel in relationdict.items():
        # print(rel) 
        r1 = (rel['r1'])
        r2 = (rel['r2'])
        rtype = rel['type']

        
        if not r1 or not r2:
            continue
        unique_relationtypes.add(rtype)
        
        r1main, r1gene, r1type = None, None, None
        try: 
            r1main = entitydict[str(r1)]['accession']
            if r1main == 'None' or r1main == None:
                continue
            r1type = entitydict[str(r1)]['type']
        except:
            print("Cannot find {} in list entities".format(r1))

        if not r1main or not r1type:
            continue

        if '#' in str(r1): # case of variant 
            try:
                r1geneid = str(entitydict[r1]['gene_ids'][0])
                r1gene = entitydict[r1geneid]['accession']
                if r1gene == 'None' or r1gene == None:
                    continue
                r1genetype = entitydict[r1geneid]['type']
                # unique_entitytypes.add(r1genetype)
                # print("Found gene {} associated with variant {}".format(r1geneid, r1main))
            except:
                print("Cannot find gene {} associated with variant {} in list entities".format(r1geneid, r1main))

        r2main, r2gene, r2type = None, None, None 
        try:
            r2main = entitydict[str(r2)]['accession']
            if r2main == 'None' or r2main == None:
                continue
            r2type = entitydict[str(r2)]['type']
        except:
            print("Cannot find {} in list entities".format(r2))
        
        if not r2main or not r2type:
            continue
        
        if "#" in str(r2): # in case of variant  
            try: 
                r2geneid = str(entitydict[r2]['gene_ids'][0])
                r2gene = entitydict[r2geneid]['accession']
                if r2gene == None or r2gene == 'None':
                    continue
                r2genetype = entitydict[r2geneid]['type']
                # unique_entitytypes.add(r2genetype)
                # print("Found gene {} associated with variant {}".format(r2geneid, r2main))
            except:
                print("Cannot find gene {} associated with variant {} in list entities".format(r2geneid, r2main))

        
        
        if r1gene: 
            r1main = r1gene
            r1type = r1genetype
        if r2gene:
            r2main = r2gene
            r2type = r2genetype 

        unique_entitytypes.add(r1type)
        unique_entitytypes.add(r2type)

        triplet = (r1main, rtype, r2main)

        scores = rel['score']
        scores = [float(x) for x in scores if float(x) <= 1.0]
        ascore = sum(scores)/len(scores)
        # scorecount  = len(scores)

        if triplet not in refinedrelations:
            # print(triplet)
            refinedrelations[triplet] = {
                'score': ascore,
                'pmid_count': len(rel['pmids']),
                'r1type': r1type,
                'r2type': r2type,
                'pmids': set(rel['pmids'])
            }
        else:
            refinedrelations[triplet]['pmids'] = refinedrelations[triplet]['pmids'].union(set(rel['pmids']))
            refinedrelations[triplet]['pmid_count'] = len(refinedrelations[triplet]['pmids'])
            refinedrelations[triplet]['score'] = ((refinedrelations[triplet]['score'])*len(refinedrelations[triplet]['pmids']) + \
                                                  ascore*len(scores))/(len(refinedrelations[triplet]['pmids']) + len(scores))
            # print("Append {} to set {}".format(rel['pmids'], refinedrelations[triplet]['pmids']))
            # refinedrelations[triplet]['score'] = (refinedrelations[triplet]['score'] + ascore)/2  # average score

        
        # if r1gene:
        #     genesWithVariants.add(r1gene)
        #     r1genetriplet = (r1gene, rtype, r2main)
        #     if r1genetriplet not in refinedrelations:
        #         refinedrelations[r1genetriplet] = {
        #             'score': ascore,
        #             'pmids': set(rel['pmids']), # originally a set 
        #             'r1type': r1genetype,
        #             'r2type': r2type
        #         }
        #     else:
        #         refinedrelations[r1genetriplet]['pmids'] = refinedrelations[r1genetriplet]['pmids'].union(set(rel['pmids']))
        #         # print("Append {} to set {}".format(rel['pmids'], refinedrelations[r1genetriplet]['pmids']))
        #         # refinedrelations[r1genetriplet]['score'] = (refinedrelations[r1genetriplet]['score'] + ascore)/2  # average score

        # if r2gene:
        #     genesWithVariants.add(r2gene)
        #     r2genetriple = (r1main, rtype, r2gene)
        #     if r2genetriple not in refinedrelations:
        #         refinedrelations[r2genetriple] = {
        #             'score': ascore,
        #             'pmids': set(rel['pmids']), # originally a set 
        #             'r1type': r1type,
        #             'r2type': r2genetype
        #         }
        #     else: 
        #         refinedrelations[r2genetriple]['pmids'] = refinedrelations[r2genetriple]['pmids'].union(set(rel['pmids']))
        #         # print("Append {} to set {}".format(rel['pmids'], refinedrelations[r2genetriple]['pmids']))
        #         # refinedrelations[r2genetriple]['score'] = (refinedrelations[r2genetriple]['score'] + ascore)/2  # average score



    # for pairid, pairinfo in refinedrelations.items():
    #     print(pairid)
    #     print(pairinfo['score'])
    #     print(pairinfo['pmids'])
    #     print(pairinfo['r1type'], pairinfo['r2type'])
        # print("="*50)
        
    print(unique_relationtypes)
    print(unique_entitytypes)

    return refinedrelations, unique_entitytypes, unique_relationtypes, genesWithVariants

In [6]:
refinedrelations, unique_entitytypes, unique_relationtypes, genesWithVariants = prepareForVisualization(entities, relations)

Cannot find 28913 in list entities
Cannot find 825516 in list entities
Cannot find 53902 in list entities
Cannot find 53901 in list entities
Cannot find 53901 in list entities
Cannot find 53901 in list entities
Cannot find 53901 in list entities
Cannot find 53902 in list entities
Cannot find 53902 in list entities
Cannot find 53902 in list entities
Cannot find 6200 in list entities
Cannot find 100358676 in list entities
Cannot find gene 22941 associated with variant @VARIANT_rs117985268 in list entities
Cannot find gene 22941 associated with variant @VARIANT_rs117985268 in list entities
Cannot find 22941 in list entities
Cannot find 22941 in list entities
Cannot find 170744 in list entities
Cannot find gene u associated with variant @VARIANT_rs11190870 in list entities
Cannot find 28927 in list entities
Cannot find 6124 in list entities
Cannot find gene 339622 associated with variant @VARIANT_rs928264 in list entities
Cannot find gene 339622 associated with variant @VARIANT_rs928264 in

In [7]:
print(len(refinedrelations))

347798


In [8]:
count = 0
for rel, data in refinedrelations.items():
    if '@CHEMICAL_Homogentisic_Acid' in rel and '@DISEASE_Alkaptonuria' in rel:
        print(rel, data)

('@CHEMICAL_Homogentisic_Acid', 'Association', '@DISEASE_Alkaptonuria') {'score': 0.8084896226415097, 'pmid_count': 106, 'r1type': 'chemical', 'r2type': 'disease', 'pmids': {'16476288', '34715699', '36684325', '16078090', '17044384', '40430165', '33655752', '19085367', '10456213', '32822600', '39312502', '15931605', '33344320', '21874298', '5559277', '24876668', '37025736', '32348369', '24781848', '38487984', '7821069', '9870204', '32875021', '39836193', '33057760', '39273071', '36465977', '25772318', '34714010', '39435171', '21308630', '16085442', '2835867', '31228435', '6503572', '34941093', '36985595', '10771134', '20952450', '31296884', '32305737', '3810644', '9056215', '39351877', '13770605', '405402', '8071207', '31609457', '30412682', '8767836', '35822095', '25182962', '21876398', '23430917', '17664775', '28158906', '2914385', '30054539', '24373924', '35152492', '8188247', '39707148', '40001497', '37404676', '33469937', '34504318', '25233259', '32395409', '27734648', '16788736',

In [9]:
unique_relationtypes.add("Multitype")

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import networkx as nx

def component_with_node(G, node):
    if node not in G:
        raise ValueError(f"Node {node} not found in graph")
    comp_nodes = nx.node_connected_component(G, node)
    return G.subgraph(comp_nodes).copy()

def construct_relation_graph_with_legend(relation_dict, entity_types, relation_types, threshold, genesWithVariants = None):
    G = nx.Graph()

    # Generate color maps
    cmap_entities = cm.get_cmap('tab20b', len(entity_types))
    cmap_relations = cm.get_cmap('Dark2', len(relation_types))

    entity_color_map = {etype: mcolors.to_hex(cmap_entities(i)) for i, etype in enumerate(sorted(entity_types))}
    relation_color_map = {rtype: mcolors.to_hex(cmap_relations(i)) for i, rtype in enumerate(sorted(relation_types))} 

    multitype_edge_color = mcolors.to_rgba(relation_color_map.get("Multitype", "#000000"))

    mr, mg, mb = [int(c * 255) for c in multitype_edge_color[:3]]
    multitype_edge_color_str = f"rgba({mr},{mg},{mb})"

    # diseasecolor =  mcolors.to_rgba(entity_color_map.get("Disease", '#ff0000'))
    # dr, dg, db = [int(c * 255) for c in diseasecolor[:3]] 


    G.add_node('@DISEASE_Alkaptonuria', label='@DISEASE_Alkaptonuria', size=30, color=entity_color_map.get("Disease", "#ff0000"))
    # G.nodes['@DISEASE_Alkaptonuria']['viz'] = {'color': {'r': dr, 'g': dg, 'b':db}}

    for (e1, rel_type, e2), data in relation_dict.items():
        if not e1 or not e2:
            continue
        if rel_type == "Comparison" or rel_type == "None" or rel_type == None: # exclude comparison 
            print("Excluding relation {} {} {}".format(e1, rel_type, e2))
            continue

        conf_score = data.get("score", 0.0)
        pmids = list((data.get("pmids", {})))
        strlist_pmids = [str(pmid) for pmid in pmids]
        str_pmids = set(strlist_pmids)
        str_pmids = list(str_pmids)
        if not str_pmids:
            continue

        type1 = data.get("r1type", "Unknown")
        type2 = data.get("r2type", "Unknown")

        edge_color_rgb = mcolors.to_rgba(relation_color_map.get(rel_type, "#000000"))

        rr, rg, rb = [int(c * 255) for c in edge_color_rgb[:3]]
        rgb_str = f"rgba({rr},{rg},{rb})"
        
        if conf_score > threshold or (len(pmids) >= 3 and conf_score >= 0.5):
            if G.has_edge(e1, e2):
                existing_pmids = G[e1][e2].get('pmidlist', {}) 
                oldlen = len(existing_pmids)
                for pimd in str_pmids:
                    existing_pmids.add(pimd)
                G[e1][e2]['pmid_count'] = len(existing_pmids)
                existing_score = G[e1][e2].get('score', 0.0)
                ave_score = (existing_score*oldlen + conf_score*len(str_pmids))/(oldlen + len(str_pmids))
                G[e1][e2]['confident'] = ave_score # average score
                oldrels = G[e1][e2].get('relation',{})
                print(e1, e2)
                print(G.get_edge_data(e1, e2))
                print(oldrels.type())
                if rel_type  not in oldrels:
                    # print(oldrels)
                    oldrels = oldrels.add(rel_type)
                    G[e1][e2]['relation'] = oldrels
                    G[e1][e2]['color'] = multitype_edge_color_str
                # G[e1][e2]['viz'] = {'color': {'r': mr, 'g': mg, 'b': mb}}
                weight = 1 - 2**(-(len(existing_pmids)))
                G[e1][e2]['score'] = weight
            else:
                if e1 in genesWithVariants:
                    G.add_node(e1, label=e1, size=15, type=type1, color=entity_color_map.get(type1, "#999999"))
                else:
                    G.add_node(e1, label=e1, color=entity_color_map.get(type1, "#999999"), type=type1)

                    # e1color =  mcolors.to_rgba(entity_color_map.get(type1, "#999999"))
                    # e1r, e1g, e1b = [int(c * 255) for c in e1color[:3]]
                    # G.nodes[e1]['viz'] = {'color': {'r': e1r, 'g': e1g, 'b': e1b}}
                if e2 in genesWithVariants:
                    G.add_node(e2, label=e2, size=15, type=type2, color=entity_color_map.get(type2, "#999999"))
                else:
                    G.add_node(e2, label=e2, color=entity_color_map.get(type2, "#999999"), type=type2)
                    # e2color =  mcolors.to_rgba(entity_color_map.get(type2, "#999999"))
                    # e2r, e2g, e2b = [int(c * 255) for c in e2color[:3]]
                    # G.nodes[e2]['viz'] = {'color': {'r': e2r, 'g': e2g, 'b': e2b}} 
                
                weight = 1 - 2**(-(len(str_pmids))) 
                pmidcount = len(str_pmids)
                # if len(str_pmids) > 3:
                #     str_pmids = list(str_pmids)
                #     pmidslable = str_pmids[0] + "|" + str_pmids[1] + "|" + str_pmids[2]
                # else:
                #     pmidslable = str_pmids[0]
                #     for strpmid in str_pmids[1:]:
                #         pmidslable + "|" + strpmid
                
                G.add_edge(e1, e2, relation={rel_type}, score = weight, confident = conf_score, \
                           pmid_count=pmidcount, pmidlist=set(str_pmids), color=rgb_str)
                # G[e1][e2]['viz'] = {'color': {'r': rr, 'g': rg, 'b': rb}}
        
    G =  component_with_node(G, '@DISEASE_Alkaptonuria')
    return G, entity_color_map, relation_color_map


In [13]:
extendGraph, entity_color, relation_color = construct_relation_graph_with_legend(refinedrelations, unique_entitytypes, unique_relationtypes, 0.7, genesWithVariants)
print(extendGraph)

@CHEMICAL_Homogentisic_Acid @CHEMICAL_Tyrosine
{'relation': {'Negative_Correlation'}, 'score': 0.998046875, 'confident': 0.8398769074675325, 'pmid_count': 77, 'pmidlist': {'24684596', '31722428', '17640269', '6886675', '2771520', '27590860', '27943071', '1185188', '14767996', '16197918', '25681086', '16730004', '11472920', '24193123', '14973036', '36465977', '31601906', '25568259', '31578365', '22100375', '18452231', '12052898', '36717', '8885416', '11073718', '12939152', '18266739', '24794033', '32305737', '26474772', '19028908', '6563016', '37443717', '36813208', '38314636', '1943688', '37175552', '35822095', '2167054', '25182962', '23430917', '22980205', '6109601', '8188247', '7567791', '28637410', '38996180', '3994376', '8382628', '23346079', '34047349', '32756361', '33920006', '32395409', '24924728', '37697495', '19862842', '20694448', '9734339', '29348338', '24816780', '24772955', '16216560', '24952314', '8782815', '23105706', '23879342', '10467142', '7753441', '15327392', '21104

/var/folders/01/v0p4sfw945j32p3yvg67p6cc0000gn/T/ipykernel_69696/2478315036.py:15: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap_entities = cm.get_cmap('tab20b', len(entity_types))
/var/folders/01/v0p4sfw945j32p3yvg67p6cc0000gn/T/ipykernel_69696/2478315036.py:16: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap_relations = cm.get_cmap('Dark2', len(relation_types))


TypeError: argument of type 'NoneType' is not iterable

In [ ]:
from pyvis.network import Network

def pyvis_subgraph_html(Gsub, fname, max_nodes=3000, max_edges=30000):
    H = Gsub.copy()

    # If still too big, slice by degree
    if H.number_of_nodes() > max_nodes:
        deg = dict(H.degree())
        keep_nodes = set(sorted(deg, key=deg.get, reverse=True)[:max_nodes])
        H = H.subgraph(keep_nodes).copy()

    # If still too many edges, keep top edges by (deg(u)+deg(v))
    if H.number_of_edges() > max_edges:
        deg = dict(H.degree())
        edges_sorted = sorted(H.edges(), key=lambda e: deg[e[0]]+deg[e[1]], reverse=True)[:max_edges]
        H = H.edge_subgraph(edges_sorted).copy()

    # Compute positions once and fix physics
    pos = nx.spring_layout(H, k=0.3, iterations=200, seed=42)

    net = Network(height="850px", width="100%", notebook=False, directed=False)
    for n, d in H.nodes(data=True):
        x, y = pos[n]
        net.add_node(n,
                     label=str(d.get("label", n)) if len(str(n)) < 40 else str(n)[:37]+"…",
                     title=str(d),
                     x=float(x*1000), y=float(y*1000), physics=False)
        

    for u, v, d in H.edges(data=True):
        net.add_edge(u, v, title=str(d))
    
    for e in net.edges:
        rel = Gsub[e['from']][e['to']]['relation']
        weight = Gsub[e['from']][e['to']]['score']
        pmid_list = Gsub[e['from']][e['to']]['pmidlist']
        numpmid = Gsub[e['from']][e['to']]['pmid_count']
        e['title'] = f"{rel} \n weight: {weight:.2f} \n {numpmid} PMIDs: {pmid_list}..."
        e['color'] = Gsub[e['from']][e['to']]['color']

    for n in net.nodes:
        node_data = Gsub.nodes[n['id']]
        n['color'] = node_data.get('color', '#999')
        n['title'] = f"Type: {node_data.get('type', 'Unknown')}"

    net.set_options("""
    var options = {
      "physics": { "enabled": false },
      "nodes": { "shape": "dot", "scaling": { "min": 1, "max": 20 } },
      "edges": { "smoothArithmeticError": false }
    }
    """)
    net.write_html(fname)

In [ ]:
center = "@DISEASE_Alkaptonuria"
radius = 1
ego = nx.ego_graph(extendGraph, center, radius=radius)
print(ego)
pyvis_subgraph_html(ego, "ego_alcaptonuria_r1.html", max_nodes=4000, max_edges=40000)


In [ ]:
import networkx as nx
from pyvis.network import Network

def visualize_communitilized(G): # pass your full graph (undirected)

    # Communities (no extra deps)
    comms = list(nx.algorithms.community.greedy_modularity_communities(G))
    comm_id = {n:i for i,c in enumerate(comms) for n in c}
    print(f"Found {len(comm_id)} communities")

    # Build community-level graph (nodes=communities, edge weight=cross edges count)
    C = nx.Graph()
    C.add_nodes_from(range(len(comms)))
    for u,v in G.edges():
        cu, cv = comm_id[u], comm_id[v]
        if cu != cv:
            C.add_edge(cu, cv, weight=C.get_edge_data(cu, cv, {}).get('weight', 0) + 1)

    # PyVis (small, fast)
    netC = Network(height="800px", width="100%", notebook=False, directed=False)
    for i, c in enumerate(comms):
        netC.add_node(i, label=f"Comm {i} (n={len(c)})", title=f"{len(c)} nodes")

    for u,v,d in C.edges(data=True):
        netC.add_edge(u, v, value=float(d.get("weight", 1)))

    netC.set_options("""
    var options = {
    physics: { enabled: false, barnesHut: { gravitationalConstant: -30000, springLength: 120 } },
    nodes: { shape: "dot", scaling: { min: 3, max: 30 } },
    edges: { smooth: false }
    }
    """)
    netC.write_html("overview_communities.html")


In [ ]:
visualize_communitilized(extendGraph) 

In [ ]:
def save_html_graph(graph, entity_color_map, relation_color_map, output_html="original.html"):
    net = Network(height="1946px", width="100%", notebook=False)
    net.from_nx(graph)
    # Customize tooltips and edge styles
    for e in net.edges:
        rel = graph[e['from']][e['to']]['relation']
        weight = graph[e['from']][e['to']]['score']
        pmid_list = graph[e['from']][e['to']]['pmidlist']
        numpmid = graph[e['from']][e['to']]['pmid_count']
        e['title'] = f"{rel} \n weight: {weight:.2f} \n {numpmid} PMIDs: {pmid_list}..."
        e['color'] = graph[e['from']][e['to']]['color']

    for n in net.nodes:
        node_data = graph.nodes[n['id']]
        n['color'] = node_data.get('color', '#999')
        n['title'] = f"Type: {node_data.get('type', 'Unknown')}"

    net.set_options("""
    var options = {
      "nodes": {
        "font": { "size": 20 },
        "scaling": { "min": 10, "max": 30 }
      },
      "edges": {
        "smooth": true
      },
      "physics": {
            "enabled": true,
            "stabilization": {
            "enabled": true,
            "iterations": 500,
            "fit": true
            }
        }
    }
    """)
    net.write_html(output_html)

    # Inject a legend block manually
    with open(output_html, "r", encoding="utf-8") as f:
        html = f.read()

    # Build legend HTML
    def dot(color):
        return f'<span style="display:inline-block;width:12px;height:12px;border-radius:6px;background:{color};margin-right:6px;"></span>'

    legend_html = "<div style='position:absolute;top:10px;right:10px;background:white;padding:10px;border:1px solid #ccc;border-radius:5px;font-family:sans-serif;font-size:14px;'>"
    legend_html += "<b>Legend</b><br><br><u>Entity Types:</u><br>"
    for etype, color in entity_color_map.items():
        legend_html += f"{dot(color)} {etype}<br>"

    legend_html += "<br><u>Relation Types:</u><br>"
    for rtype, color in relation_color_map.items():
        legend_html += f"{dot(color)} {rtype}<br>"

    legend_html += "</div>"

    # Inject just before closing </body> tag
    html = html.replace("</body>", legend_html + "\n</body>")

    # Save modified file
    with open(output_html, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Graph with legend saved to {output_html}")

## Statistics for Extended Graph

In [ ]:
dcount, ccount, gcount, other = 0, 0, 0, 0

for anode in extendGraph.nodes:
    if '@DISEASE' in anode:
        dcount += 1
    elif '@CHEMICAL' in anode:
        ccount += 1
    elif '@GENE' in anode:
        gcount += 1
    else:
        print(anode)
        other += 1
print(f'{dcount} diseases \n{ccount} chemicals \n{gcount} genes\n{other} others')


# Build high-confidence graph

In [ ]:
import networkx as nx

# Suppose G is your existing graph
highconfgraph = nx.Graph()

# Keep nodes and edges that satisfy the condition
for u, v, data in extendGraph.edges(data=True):
    if data.get("pmid_count", 0) > 2:
        highconfgraph.add_edge(u, v, **data)

# Preserve node attributes from the original graph
for n, d in extendGraph.nodes(data=True):
    if n in highconfgraph:
        highconfgraph.nodes[n].update(d)

highconfgraph = component_with_node(highconfgraph, '@DISEASE_Alkaptonuria')

print(f"Original: {extendGraph.number_of_nodes()} nodes, {extendGraph.number_of_edges()} edges")
print(f"Filtered: {highconfgraph.number_of_nodes()} nodes, {highconfgraph.number_of_edges()} edges")


In [ ]:
save_html_graph(highconfgraph, entity_color, relation_color, "../HTMLs/highconfidence.html")

## Statistic highconfgraph

In [ ]:
dcount, ccount, gcount, other = 0, 0, 0, 0

for anode in highconfgraph.nodes:
    if '@DISEASE' in anode:
        dcount += 1
    elif '@CHEMICAL' in anode:
        ccount += 1
    elif '@GENE' in anode:
        gcount += 1
    else:
        print(anode)
        other += 1
print(f'{dcount} diseases \n{ccount} chemicals \n{gcount} genes\n{other} others')
